# SKKB - Final Analysis

Computations for `skkb_exp_001_baseline.yaml` using the SKKB final analysis plan.

Input checkpoint:

`checkpoints/evals_skkb_exp_001_baseline_medium.csv`

This notebook is intentionally separate from `skkb_001_results_viewer.ipynb`. It keeps the final-report computations in one top-to-bottom flow:

- dataset/scope breakdown
- deterministic retrieval and reranker metrics
- judge weighted average and gates
- operational verdicts and affected teams
- candidate-pool, answer, language, and reliability analysis
- calibration against expert signals
- stakeholder tables and manual-review queue

## Parameters

Set these if you want to inspect another checkpoint or use a different gate threshold.

In [0]:
RUN_ON_DBX = False

YAML_FILE_NAME = "skkb_exp_001_baseline"
REASONING_EFFORT = "medium"
CHECKPOINT_SUFFIX = ""
CHECKPOINT_CSV = f"checkpoints/evals_{YAML_FILE_NAME}_{REASONING_EFFORT}{CHECKPOINT_SUFFIX}.csv"

PASS_THRESHOLD = 1.4
NDCG_K = 10
LOW_RANK_THRESHOLD = 5

EXPORT_ENRICHED = False

## Imports And Load

The checkpoint loader repairs known ragged-row shapes from append-style CSV checkpointing.

In [0]:
import ast
import json
import math
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:
    def display(obj=None, *args, **kwargs):
        print(obj)

    def Markdown(text):
        return text

for module_dir in (
    Path.cwd(),
    Path.cwd() / "notebooks/skkb",
    Path.cwd() / "ds-evals/notebooks/skkb",
):
    if (module_dir / "skkb_checkpoint.py").exists():
        sys.path.insert(0, str(module_dir))
        break

from skkb_checkpoint import read_checkpoint_csv


DIMENSION_WEIGHTS = {
    "query_clarity": 1.0,
    "case_scope_fit": 1.0,
    "selection_semantic_relevance": 1.5,
    "selected_context_sufficiency": 1.5,
    "optimal_retrieved_context_adequacy": 1.5,
    "answer_expected_alignment": 2.0,
    "answer_groundedness": 2.0,
    "language_compliance": 1.0,
}

FREE_TEXT_JUDGE_FIELDS = [
    "overall_explanation",
    "candidate_pool_content_gap_description",
    "expected_answer_summary_with_optimal_context",
    "retrieval_improvement_suggestion",
    "reranker_improvement_suggestion",
    "bot_improvement_suggestion",
    "kb_improvement_suggestion",
    "test_case_improvement_suggestion",
]
FREE_TEXT_JUDGE_FIELDS += [f"{dim}_reasoning" for dim in DIMENSION_WEIGHTS]

LIST_JUDGE_FIELDS = [
    "issue_categories",
    "affected_teams",
    "extra_or_distracting_enums",
    "optimal_enum_selection",
    "unavailable_facts_in_selected_context",
    "missing_facts",
    "hallucinated_claims",
    "bot_language_issue_evidence",
]


def resolve_checkpoint_path(path: str | Path) -> Path:
    path = Path(path)
    if path.is_absolute():
        candidates = [path]
    else:
        candidates = [
            path,
            Path.cwd() / path,
            Path.cwd() / "notebooks/skkb" / path,
            Path.cwd() / "ds-evals/notebooks/skkb" / path,
        ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError(f"Could not find checkpoint CSV. Searched:\n{searched}")


checkpoint_path = resolve_checkpoint_path(CHECKPOINT_CSV)
df = read_checkpoint_csv(checkpoint_path)

print(f"checkpoint: {checkpoint_path}")
print(f"rows: {len(df):,}   columns: {len(df.columns):,}")


## Parsing Helpers

The checkpoint stores lists and dicts as JSON-ish strings. These helpers keep the downstream computations deterministic and resilient to empty cells.

In [0]:
TRUE_VALUES = {"true", "1", "1.0", "yes", "y"}
FALSE_VALUES = {"false", "0", "0.0", "no", "n", ""}


def parse_jsonish(value, default=None):
    if isinstance(value, (list, dict, tuple)):
        return value
    if value is None:
        return default
    if isinstance(value, float) and pd.isna(value):
        return default
    if not isinstance(value, str):
        return default
    text = value.strip()
    if not text:
        return default
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return default


def parse_list(value) -> list:
    parsed = parse_jsonish(value, default=[])
    return parsed if isinstance(parsed, list) else []


def parse_dict(value) -> dict:
    parsed = parse_jsonish(value, default={})
    return parsed if isinstance(parsed, dict) else {}


def parse_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None:
        return False
    if isinstance(value, float) and pd.isna(value):
        return False
    text = str(value).strip().lower()
    if text in TRUE_VALUES:
        return True
    if text in FALSE_VALUES:
        return False
    return False


def clean_str(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value).strip()


def is_empty_context(value) -> bool:
    text = clean_str(value)
    if text.lower() in {"", "[]", "{}", "null", "none"}:
        return True
    parsed = parse_jsonish(text, default=None)
    if isinstance(parsed, (list, dict)):
        return len(parsed) == 0
    return False


def unique_preserve_order(items) -> list:
    seen = set()
    out = []
    for item in items:
        key = str(item)
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out


def safe_float(value, default=np.nan):
    try:
        if value is None or (isinstance(value, float) and pd.isna(value)):
            return default
        return float(value)
    except (TypeError, ValueError):
        return default


def expected_weight_map(raw_weights, expected_enums: list[str]) -> dict[str, float]:
    parsed_dict = parse_dict(raw_weights)
    if parsed_dict:
        return {
            enum_id: max(safe_float(parsed_dict.get(enum_id), 1.0), 0.0)
            for enum_id in expected_enums
        }

    parsed_list = parse_list(raw_weights)
    if parsed_list and all(isinstance(x, (int, float, str)) for x in parsed_list):
        return {
            enum_id: max(safe_float(parsed_list[i], 1.0), 0.0)
            for i, enum_id in enumerate(expected_enums)
        }

    return {enum_id: 1.0 for enum_id in expected_enums}


def tool_names(value) -> list[str]:
    names = []
    for item in parse_list(value):
        if isinstance(item, dict) and clean_str(item.get("name")):
            names.append(clean_str(item.get("name")))
        elif isinstance(item, str) and item.strip():
            names.append(item.strip())
    return names


def agent_names(value) -> list[str]:
    return [clean_str(item) for item in parse_list(value) if clean_str(item)]

## Normalize Scores And Parsed Columns

These parsed `_...` columns are used by all later exact-set and validation computations.

In [0]:
required_score_cols = [f"{dim}_score" for dim in DIMENSION_WEIGHTS]
missing_score_cols = [col for col in required_score_cols if col not in df.columns]
if missing_score_cols:
    raise KeyError(f"Missing judge dimension score columns: {missing_score_cols}")

for col in required_score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

if "expert_score" in df.columns:
    df["expert_score"] = pd.to_numeric(df["expert_score"], errors="coerce")
if "enum_relevance_score" in df.columns:
    df["enum_relevance_score"] = pd.to_numeric(df["enum_relevance_score"], errors="coerce")

LIST_COLUMNS = [
    "expected_enums",
    "retrieved_enum_ids",
    "reranked_enum_ids",
    "reranker_raw_selected_ids",
    "reranker_valid_selected_ids",
    "reranker_invalid_selected_ids",
    "reranker_unselected_context_ids",
    "reranker_selection_violations",
    "issue_categories",
    "affected_teams",
    "extra_or_distracting_enums",
    "optimal_enum_selection",
    "unavailable_facts_in_selected_context",
    "missing_facts",
    "hallucinated_claims",
    "bot_language_issue_evidence",
    "categories_list",
]

for col in LIST_COLUMNS:
    if col in df.columns:
        df[f"_{col}"] = df[col].apply(parse_list)
    else:
        df[f"_{col}"] = [[] for _ in range(len(df))]

df["_expected_enum_weights"] = df.apply(
    lambda row: expected_weight_map(
        row.get("expected_enums_weights"),
        row.get("_expected_enums", []),
    ),
    axis=1,
)
df["_tool_names"] = df["tools_called"].apply(tool_names) if "tools_called" in df.columns else [[] for _ in range(len(df))]
df["_agent_names"] = df["agents_called"].apply(agent_names) if "agents_called" in df.columns else [[] for _ in range(len(df))]

if "candidate_pool_content_gap_identified" in df.columns:
    df["candidate_pool_content_gap_identified_bool"] = df["candidate_pool_content_gap_identified"].apply(parse_bool)
else:
    df["candidate_pool_content_gap_identified_bool"] = False

df["error_bool"] = df["error"].apply(parse_bool) if "error" in df.columns else False

## Dataset And Scope Breakdown

Use judge `case_scope`, deterministic `query_scope`, and their mismatch to decide which rows are clean KB-content evidence.

In [0]:
def count_share(series: pd.Series, name: str = "count") -> pd.DataFrame:
    counts = series.fillna("(missing)").replace("", "(blank)").value_counts(dropna=False)
    return (
        counts.rename(name)
        .to_frame()
        .assign(share=lambda x: (x[name] / x[name].sum()).round(4))
    )


scope_tables = {}
if "case_scope" in df.columns:
    scope_tables["case_scope_distribution"] = count_share(df["case_scope"], "rows")
    display(scope_tables["case_scope_distribution"])

if "query_scope" in df.columns:
    scope_tables["query_scope_distribution"] = count_share(df["query_scope"], "rows")
    display(scope_tables["query_scope_distribution"])

if {"case_scope", "query_scope"}.issubset(df.columns):
    scope_crosstab = pd.crosstab(
        df["query_scope"].fillna("(missing)").replace("", "(blank)"),
        df["case_scope"].fillna("(missing)").replace("", "(blank)"),
        margins=True,
    )
    display(scope_crosstab)

NON_KB_CASE_SCOPES = {"account_data_by_design", "out_of_scope", "ambiguous", "unclear_test_case", "(blank)", ""}
df["exclude_from_kb_content_conclusions"] = df["case_scope"].fillna("").isin(NON_KB_CASE_SCOPES)

print(
    "rows excluded from KB-content conclusions: "
    f"{int(df['exclude_from_kb_content_conclusions'].sum())}/{len(df)} "
    f"({df['exclude_from_kb_content_conclusions'].mean():.1%})"
)

## Deterministic Retrieval And Reranker Metrics

These are exact computations from `expected_enums`, `retrieved_enum_ids`, `reranked_enum_ids`, and `expected_enums_weights`.

In [0]:
def rank_map(items: list[str]) -> dict[str, int]:
    ranks = {}
    for idx, item in enumerate(items, start=1):
        ranks.setdefault(str(item), idx)
    return ranks


def retrieval_recall(row) -> float:
    expected = set(row["_expected_enums"])
    if not expected:
        return np.nan
    retrieved = set(row["_retrieved_enum_ids"])
    return len(expected & retrieved) / len(expected)


def weighted_retrieval_recall(row) -> float:
    expected = set(row["_expected_enums"])
    if not expected:
        return np.nan
    weights = row["_expected_enum_weights"]
    denom = sum(weights.get(enum_id, 1.0) for enum_id in expected)
    if denom <= 0:
        return np.nan
    retrieved = set(row["_retrieved_enum_ids"])
    numerator = sum(weights.get(enum_id, 1.0) for enum_id in expected & retrieved)
    return numerator / denom


def best_expected_mrr(row) -> float:
    expected = set(row["_expected_enums"])
    if not expected:
        return np.nan
    ranks = rank_map(row["_retrieved_enum_ids"])
    hits = [1 / ranks[enum_id] for enum_id in expected if enum_id in ranks]
    return max(hits) if hits else 0.0


def mean_expected_reciprocal_rank(row) -> float:
    expected = list(row["_expected_enums"])
    if not expected:
        return np.nan
    ranks = rank_map(row["_retrieved_enum_ids"])
    return sum(1 / ranks[enum_id] if enum_id in ranks else 0.0 for enum_id in expected) / len(expected)


def ndcg_at_k(row, k: int = NDCG_K) -> float:
    expected = set(row["_expected_enums"])
    if not expected:
        return np.nan
    weights = row["_expected_enum_weights"]
    retrieved = row["_retrieved_enum_ids"][:k]

    def dcg(relevances: list[float]) -> float:
        return sum(rel / math.log2(rank + 1) for rank, rel in enumerate(relevances, start=1))

    actual_rels = [weights.get(enum_id, 0.0) if enum_id in expected else 0.0 for enum_id in retrieved]
    ideal_rels = sorted([weights.get(enum_id, 1.0) for enum_id in expected], reverse=True)[:k]
    ideal = dcg(ideal_rels)
    if ideal <= 0:
        return np.nan
    return dcg(actual_rels) / ideal


def selection_prf(row) -> pd.Series:
    selected = set(row["_reranked_enum_ids"])
    expected = set(row["_expected_enums"])
    if not selected and not expected:
        return pd.Series({"selection_precision": np.nan, "selection_recall": np.nan, "selection_f1": np.nan})
    tp = len(selected & expected)
    precision = tp / len(selected) if selected else np.nan
    recall = tp / len(expected) if expected else np.nan
    if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    elif precision == 0 or recall == 0:
        f1 = 0.0
    else:
        f1 = np.nan
    return pd.Series({"selection_precision": precision, "selection_recall": recall, "selection_f1": f1})


def reranker_conditional_recall(row) -> float:
    expected_retrieved = set(row["_expected_enums"]) & set(row["_retrieved_enum_ids"])
    if not expected_retrieved:
        return np.nan
    selected = set(row["_reranked_enum_ids"])
    return len(expected_retrieved & selected) / len(expected_retrieved)


df["retrieval_recall"] = df.apply(retrieval_recall, axis=1)
df["weighted_retrieval_recall"] = df.apply(weighted_retrieval_recall, axis=1)
df["best_expected_mrr"] = df.apply(best_expected_mrr, axis=1)
df["mean_expected_reciprocal_rank"] = df.apply(mean_expected_reciprocal_rank, axis=1)
df[f"retriever_ndcg_at_{NDCG_K}"] = df.apply(lambda row: ndcg_at_k(row, NDCG_K), axis=1)

df[["selection_precision", "selection_recall", "selection_f1"]] = df.apply(selection_prf, axis=1)
df["reranker_conditional_recall"] = df.apply(reranker_conditional_recall, axis=1)

df["_expected_enums_missed_by_retriever"] = df.apply(
    lambda row: sorted(set(row["_expected_enums"]) - set(row["_retrieved_enum_ids"])),
    axis=1,
)
df["_expected_enums_missed_by_reranker"] = df.apply(
    lambda row: sorted(
        (set(row["_expected_enums"]) & set(row["_retrieved_enum_ids"]))
        - set(row["_reranked_enum_ids"])
    ),
    axis=1,
)
df["_extra_selected_enums"] = df.apply(
    lambda row: sorted(set(row["_reranked_enum_ids"]) - set(row["_expected_enums"])),
    axis=1,
)

df["expected_enums_missed_by_retriever"] = df["_expected_enums_missed_by_retriever"].apply(json.dumps)
df["expected_enums_missed_by_reranker"] = df["_expected_enums_missed_by_reranker"].apply(json.dumps)
df["extra_selected_enums"] = df["_extra_selected_enums"].apply(json.dumps)

df["invalid_selected_enum_count"] = df["_reranker_invalid_selected_ids"].apply(len)
df["reranker_empty_selection"] = df["_reranked_enum_ids"].apply(len).eq(0)
if "reranker_selected_empty" in df.columns:
    df["reranker_empty_selection"] = df["reranker_empty_selection"] | df["reranker_selected_empty"].apply(parse_bool)

df["retriever_missed_expected_count"] = df["_expected_enums_missed_by_retriever"].apply(len)
df["reranker_missed_expected_count"] = df["_expected_enums_missed_by_reranker"].apply(len)
df["extra_selected_enum_count"] = df["_extra_selected_enums"].apply(len)
df["selected_expected_exact_match"] = df.apply(
    lambda row: set(row["_reranked_enum_ids"]) == set(row["_expected_enums"]) if row["_expected_enums"] else np.nan,
    axis=1,
)

metric_summary = pd.DataFrame(
    {
        "metric": [
            "retrieval_recall",
            "weighted_retrieval_recall",
            "best_expected_mrr",
            "mean_expected_reciprocal_rank",
            f"retriever_ndcg_at_{NDCG_K}",
            "selection_precision",
            "selection_recall",
            "selection_f1",
            "reranker_conditional_recall",
        ],
        "mean": [
            df["retrieval_recall"].mean(),
            df["weighted_retrieval_recall"].mean(),
            df["best_expected_mrr"].mean(),
            df["mean_expected_reciprocal_rank"].mean(),
            df[f"retriever_ndcg_at_{NDCG_K}"].mean(),
            df["selection_precision"].mean(),
            df["selection_recall"].mean(),
            df["selection_f1"].mean(),
            df["reranker_conditional_recall"].mean(),
        ],
        "median": [
            df["retrieval_recall"].median(),
            df["weighted_retrieval_recall"].median(),
            df["best_expected_mrr"].median(),
            df["mean_expected_reciprocal_rank"].median(),
            df[f"retriever_ndcg_at_{NDCG_K}"].median(),
            df["selection_precision"].median(),
            df["selection_recall"].median(),
            df["selection_f1"].median(),
            df["reranker_conditional_recall"].median(),
        ],
        "non_null": [
            df["retrieval_recall"].notna().sum(),
            df["weighted_retrieval_recall"].notna().sum(),
            df["best_expected_mrr"].notna().sum(),
            df["mean_expected_reciprocal_rank"].notna().sum(),
            df[f"retriever_ndcg_at_{NDCG_K}"].notna().sum(),
            df["selection_precision"].notna().sum(),
            df["selection_recall"].notna().sum(),
            df["selection_f1"].notna().sum(),
            df["reranker_conditional_recall"].notna().sum(),
        ],
    }
).round(4)

display(metric_summary)

## Routing And Technical Flags

These deterministic flags should not be inferred by the LLM judge.

In [0]:
df["kb_case_without_knowledge_search"] = df.apply(
    lambda row: row.get("case_scope") == "kb_answerable" and "knowledge_search" not in row["_tool_names"],
    axis=1,
)
df["empty_kb_context_for_kb_case"] = df.apply(
    lambda row: (
        row.get("case_scope") == "kb_answerable"
        and is_empty_context(row.get("retrieved_candidates_text"))
        and is_empty_context(row.get("reranked_kb_context"))
    ),
    axis=1,
)

df["retriever_low_rank_expected_count"] = df.apply(
    lambda row: sum(
        1
        for enum_id, rank in rank_map(row["_retrieved_enum_ids"]).items()
        if enum_id in set(row["_expected_enums"]) and rank > LOW_RANK_THRESHOLD
    ),
    axis=1,
)


def deterministic_issues(row) -> list[str]:
    issues = []
    if row["retriever_missed_expected_count"] > 0:
        issues.append("retriever_miss")
    if row["retriever_low_rank_expected_count"] > 0:
        issues.append("retriever_low_rank")
    if row["reranker_missed_expected_count"] > 0:
        issues.append("reranker_miss")
    if row["invalid_selected_enum_count"] > 0:
        issues.append("reranker_invalid_output")
    if row["reranker_empty_selection"]:
        issues.append("reranker_empty_selection")
    if row["kb_case_without_knowledge_search"]:
        issues.append("routing_tool_issue")
    if row["empty_kb_context_for_kb_case"]:
        issues.append("empty_kb_context_for_kb_case")
    return issues


DETERMINISTIC_ISSUE_TEAMS = {
    "retriever_miss": "retrieval",
    "retriever_low_rank": "retrieval",
    "reranker_miss": "reranker",
    "reranker_invalid_output": "reranker",
    "reranker_empty_selection": "reranker",
    "routing_tool_issue": "routing_tooling",
    "empty_kb_context_for_kb_case": "routing_tooling",
}

df["_deterministic_issues"] = df.apply(deterministic_issues, axis=1)
df["deterministic_issues"] = df["_deterministic_issues"].apply(json.dumps)
df["_deterministic_affected_teams"] = df["_deterministic_issues"].apply(
    lambda issues: unique_preserve_order(DETERMINISTIC_ISSUE_TEAMS[i] for i in issues if i in DETERMINISTIC_ISSUE_TEAMS)
)
df["deterministic_affected_teams"] = df["_deterministic_affected_teams"].apply(json.dumps)

technical_flag_summary = pd.DataFrame(
    {
        "flag": [
            "retriever_miss",
            "retriever_low_rank",
            "reranker_miss",
            "reranker_invalid_output",
            "reranker_empty_selection",
            "routing_tool_issue",
            "empty_kb_context_for_kb_case",
        ],
        "rows": [
            (df["retriever_missed_expected_count"] > 0).sum(),
            (df["retriever_low_rank_expected_count"] > 0).sum(),
            (df["reranker_missed_expected_count"] > 0).sum(),
            (df["invalid_selected_enum_count"] > 0).sum(),
            df["reranker_empty_selection"].sum(),
            df["kb_case_without_knowledge_search"].sum(),
            df["empty_kb_context_for_kb_case"].sum(),
        ],
    }
)
technical_flag_summary["share"] = (technical_flag_summary["rows"] / len(df)).round(4)
display(technical_flag_summary)

## Judge Weighted Average

The canonical metric stays on the 0-2 scale. The percentage is display-only.

In [0]:
weight_sum = sum(DIMENSION_WEIGHTS.values())
df["judge_weighted_avg"] = sum(
    df[f"{dim}_score"] * weight for dim, weight in DIMENSION_WEIGHTS.items()
) / weight_sum
df["weighted_avg"] = df["judge_weighted_avg"]
df["judge_quality_pct"] = df["judge_weighted_avg"] / 2.0 * 100
df["judge_weighted_sum"] = df["judge_weighted_avg"] * weight_sum
df["weighted_loss_points"] = (2.0 * weight_sum) - df["judge_weighted_sum"]
df["weighted_avg_pass"] = df["judge_weighted_avg"] >= PASS_THRESHOLD

weighted_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "weighted_avg_pass_rate",
            "judge_weighted_avg_mean",
            "judge_weighted_avg_median",
            "judge_quality_pct_mean",
            "weighted_loss_points_mean",
        ],
        "value": [
            len(df),
            df["weighted_avg_pass"].mean(),
            df["judge_weighted_avg"].mean(),
            df["judge_weighted_avg"].median(),
            df["judge_quality_pct"].mean(),
            df["weighted_loss_points"].mean(),
        ],
    }
)
display(weighted_summary)

dimension_distribution = (
    df[[f"{dim}_score" for dim in DIMENSION_WEIGHTS]]
    .apply(lambda s: s.value_counts(dropna=False).reindex([0, 1, 2]).fillna(0).astype(int))
    .T
)
dimension_distribution.index = [idx.replace("_score", "") for idx in dimension_distribution.index]
dimension_distribution.columns = ["score_0", "score_1", "score_2"]
dimension_distribution["zero_rate"] = (dimension_distribution["score_0"] / len(df)).round(4)
dimension_distribution["good_rate"] = (dimension_distribution["score_2"] / len(df)).round(4)
display(dimension_distribution)

## Consistency Validation

Errors make a row unreliable for quality gates. Warnings mark suspicious but still usable combinations.

In [0]:
CZ_SK_DIACRITICS = re.compile(r"[áäčďéěíĺľňóôŕřšťúůýžÁÄČĎÉĚÍĹĽŇÓÔŔŘŠŤÚŮÝŽ]")


def text_has_cz_sk_diacritics(value) -> bool:
    if isinstance(value, list):
        return any(text_has_cz_sk_diacritics(item) for item in value)
    text = clean_str(value)
    return bool(text and CZ_SK_DIACRITICS.search(text))


def row_validation(row) -> pd.Series:
    errors = []
    warnings = []
    issue_categories = set(row["_issue_categories"])
    affected_teams = set(row["_affected_teams"])

    deterministic_retriever_miss = row["retriever_missed_expected_count"] > 0
    deterministic_reranker_miss = row["reranker_missed_expected_count"] > 0

    if deterministic_retriever_miss and "retrieval" not in affected_teams:
        warnings.append("deterministic retriever miss but retrieval not in affected_teams")

    if (
        deterministic_reranker_miss
        and row["selection_semantic_relevance_score"] == 2
        and row["selected_context_sufficiency_score"] == 2
    ):
        warnings.append("reranker missed expected enum but judge scored selection/context fully sufficient")

    if row["candidate_pool_content_gap_identified_bool"] and row["optimal_retrieved_context_adequacy_score"] == 2:
        errors.append("candidate_pool_content_gap_identified=true but optimal_retrieved_context_adequacy_score=2")

    if row["candidate_pool_content_gap_identified_bool"] and row.get("case_scope") != "kb_answerable":
        errors.append("candidate_pool_content_gap_identified=true for non-kb_answerable case")

    if "candidate_pool_content_gap" in issue_categories and "kb_editors" not in affected_teams:
        errors.append("candidate_pool_content_gap issue without kb_editors affected_team")

    if clean_str(row.get("kb_change_type")).lower() not in {"", "none", "nan"} and "kb_editors" not in affected_teams:
        warnings.append("kb_change_type set but kb_editors not in affected_teams")

    if row["answer_groundedness_score"] == 0 and len(row["_hallucinated_claims"]) == 0:
        errors.append("answer_groundedness_score=0 but hallucinated_claims is empty")

    explanation = clean_str(row.get("overall_explanation")).lower()
    explains_wrong_answer = bool(re.search(r"contradict|wrong|refus|unavailable|not available|instead|fabricat", explanation))
    if row["answer_expected_alignment_score"] == 0 and len(row["_missing_facts"]) == 0 and not explains_wrong_answer:
        warnings.append("answer_expected_alignment_score=0 but missing_facts is empty and explanation lacks clear wrong-answer rationale")

    if "answer_generation_issue" in issue_categories and row["selected_context_sufficiency_score"] == 0:
        warnings.append("answer_generation_issue with selected_context_sufficiency_score=0 may be upstream")

    if row["language_compliance_score"] < 2 and len(row["_bot_language_issue_evidence"]) == 0:
        errors.append("language_compliance_score<2 but bot_language_issue_evidence is empty")

    if "language_mismatch" in issue_categories and row["language_compliance_score"] == 2:
        errors.append("language_mismatch issue but language_compliance_score=2")

    for field in FREE_TEXT_JUDGE_FIELDS:
        if field in row and text_has_cz_sk_diacritics(row.get(field)):
            errors.append(f"judge free-text field appears non-English or copied SK/CZ evidence: {field}")
            break
    for field in LIST_JUDGE_FIELDS:
        if text_has_cz_sk_diacritics(row.get(f"_{field}", [])):
            errors.append(f"judge list field appears non-English or copied SK/CZ evidence: {field}")
            break

    primary_root_cause = clean_str(row.get("primary_root_cause"))
    if primary_root_cause == "no_issue" and len(issue_categories) > 0:
        errors.append("primary_root_cause=no_issue but issue_categories is not empty")

    if primary_root_cause not in {"", "no_issue"} and len(issue_categories) == 0:
        errors.append("primary_root_cause set but issue_categories is empty")

    improvement_fields = [
        "retrieval_improvement_suggestion",
        "reranker_improvement_suggestion",
        "bot_improvement_suggestion",
        "kb_improvement_suggestion",
        "test_case_improvement_suggestion",
    ]
    if primary_root_cause not in {"", "no_issue"} and not any(clean_str(row.get(field)) for field in improvement_fields):
        errors.append("primary_root_cause set but all improvement suggestion fields are empty")

    if len(affected_teams) == 0 and len(issue_categories) > 0:
        errors.append("issue_categories is not empty but affected_teams is empty")

    if clean_str(row.get("technical_change_type")) == "reranker_prompt" and "reranker" not in affected_teams:
        warnings.append("technical_change_type=reranker_prompt but reranker not in affected_teams")

    return pd.Series(
        {
            "_validation_errors": errors,
            "_validation_warnings": warnings,
            "validation_errors": json.dumps(errors),
            "validation_warnings": json.dumps(warnings),
            "validation_error_count": len(errors),
            "validation_warning_count": len(warnings),
        }
    )


validation_cols = df.apply(row_validation, axis=1)
df = pd.concat([df, validation_cols], axis=1)
df["judge_consistency_pass"] = df["validation_error_count"].eq(0)
df["judge_consistency_clean"] = df["validation_error_count"].eq(0) & df["validation_warning_count"].eq(0)

consistency_summary = pd.DataFrame(
    {
        "metric": ["judge_consistency_pass_rate", "judge_consistency_clean_rate", "rows_with_errors", "rows_with_warnings"],
        "value": [
            df["judge_consistency_pass"].mean(),
            df["judge_consistency_clean"].mean(),
            (df["validation_error_count"] > 0).sum(),
            (df["validation_warning_count"] > 0).sum(),
        ],
    }
)
display(consistency_summary)

validation_error_distribution = Counter(
    error
    for errors in df["_validation_errors"]
    for error in errors
)
validation_warning_distribution = Counter(
    warning
    for warnings in df["_validation_warnings"]
    for warning in warnings
)

display(pd.DataFrame(validation_error_distribution.most_common(), columns=["validation_error", "count"]))
display(pd.DataFrame(validation_warning_distribution.most_common(), columns=["validation_warning", "count"]).head(25))

## Quality Gate, Pipeline Pass, And Operational Verdict

`quality_gate` is the product-quality gate. `pipeline_pass` is stricter and only applies to `kb_answerable` cases.

In [0]:
df["quality_gate"] = (
    (df["judge_weighted_avg"] >= PASS_THRESHOLD)
    & (df["answer_expected_alignment_score"] > 0)
    & (df["answer_groundedness_score"] > 0)
    & (df["language_compliance_score"] > 0)
    & (df["case_scope_fit_score"] > 0)
    & df["judge_consistency_pass"]
)

df["pipeline_pass"] = (
    (df["case_scope"] == "kb_answerable")
    & (df["retrieval_recall"] >= 0.999)
    & (df["reranker_conditional_recall"] >= 0.999)
    & (df["selection_semantic_relevance_score"] >= 1)
    & (df["selected_context_sufficiency_score"] >= 1)
    & (df["optimal_retrieved_context_adequacy_score"] >= 1)
    & (df["answer_expected_alignment_score"] >= 1)
    & (df["answer_groundedness_score"] >= 1)
    & (df["language_compliance_score"] == 2)
    & df["judge_consistency_pass"]
)


def combined_affected_teams(row) -> list[str]:
    return unique_preserve_order(row["_affected_teams"] + row["_deterministic_affected_teams"])


df["_combined_affected_teams"] = df.apply(combined_affected_teams, axis=1)
df["combined_affected_teams"] = df["_combined_affected_teams"].apply(json.dumps)


def has_any_issue(row) -> bool:
    if row["_issue_categories"] or row["_deterministic_issues"]:
        return True
    if clean_str(row.get("primary_root_cause")) not in {"", "no_issue"}:
        return True
    if row["candidate_pool_content_gap_identified_bool"]:
        return True
    return False


def operational_verdict(row) -> str:
    teams = set(row["_combined_affected_teams"])
    if not row["judge_consistency_pass"]:
        return "judge_review"
    if "test_set" in teams and row.get("case_scope") != "kb_answerable":
        return "test_set_action"
    if row["kb_case_without_knowledge_search"] or row["empty_kb_context_for_kb_case"]:
        return "routing_tool_action"
    if len(teams) >= 2 and not row["quality_gate"]:
        return "multi_team_blocker"
    if row["retriever_missed_expected_count"] > 0 or "retrieval" in teams:
        return "retrieval_action"
    if row["reranker_missed_expected_count"] > 0 or row["invalid_selected_enum_count"] > 0 or "reranker" in teams:
        return "reranker_action"
    if row["candidate_pool_content_gap_identified_bool"]:
        return "candidate_pool_content_action"
    if "kb_editors" in teams:
        return "kb_editorial_action"
    if "bot_agent" in teams:
        return "bot_agent_action"
    if row["quality_gate"] and has_any_issue(row):
        return "pass_minor_action"
    return "pass_no_action"


df["has_any_issue"] = df.apply(has_any_issue, axis=1)
df["operational_verdict"] = df.apply(operational_verdict, axis=1)

gate_summary = pd.DataFrame(
    {
        "metric": [
            "weighted_avg_pass_rate",
            "quality_gate_pass_rate",
            "pipeline_pass_rate_all_rows",
            "pipeline_pass_rate_kb_answerable",
        ],
        "value": [
            df["weighted_avg_pass"].mean(),
            df["quality_gate"].mean(),
            df["pipeline_pass"].mean(),
            df.loc[df["case_scope"] == "kb_answerable", "pipeline_pass"].mean(),
        ],
    }
)
display(gate_summary)
display(count_share(df["operational_verdict"], "rows"))

## Executive Summary Inputs

This table contains the KPI values used by the final written summary.

In [0]:
def top_counter_values(values, n=5) -> str:
    counter = Counter(values)
    return ", ".join(f"{key} ({count})" for key, count in counter.most_common(n))


retriever_miss_counter = Counter(
    enum_id
    for enums in df["_expected_enums_missed_by_retriever"]
    for enum_id in enums
)
reranker_miss_counter = Counter(
    enum_id
    for enums in df["_expected_enums_missed_by_reranker"]
    for enum_id in enums
)
affected_team_counter = Counter(
    team
    for teams in df["_combined_affected_teams"]
    for team in teams
)

executive_kpis = pd.DataFrame(
    [
        ("evaluated_cases", len(df)),
        ("excluded_or_non_kb_scope_cases", int(df["exclude_from_kb_content_conclusions"].sum())),
        ("weighted_avg_pass_rate", df["weighted_avg_pass"].mean()),
        ("quality_gate_pass_rate", df["quality_gate"].mean()),
        ("pipeline_pass_rate_kb_answerable", df.loc[df["case_scope"] == "kb_answerable", "pipeline_pass"].mean()),
        ("judge_weighted_avg_mean", df["judge_weighted_avg"].mean()),
        ("judge_weighted_avg_median", df["judge_weighted_avg"].median()),
        ("retrieval_recall_mean", df["retrieval_recall"].mean()),
        ("weighted_retrieval_recall_mean", df["weighted_retrieval_recall"].mean()),
        ("selection_f1_mean", df["selection_f1"].mean()),
        ("candidate_pool_content_gap_rate_kb_answerable", df.loc[df["case_scope"] == "kb_answerable", "candidate_pool_content_gap_identified_bool"].mean()),
        ("language_mismatch_rate", (df["language_compliance_score"] < 2).mean()),
        ("judge_consistency_pass_rate", df["judge_consistency_pass"].mean()),
        ("top_primary_root_causes", top_counter_values(df["primary_root_cause"].fillna("(missing)").replace("", "(blank)")) if "primary_root_cause" in df.columns else ""),
        ("top_affected_teams", ", ".join(f"{team} ({count})" for team, count in affected_team_counter.most_common(5))),
        ("top_retriever_missed_enums", ", ".join(f"{enum_id} ({count})" for enum_id, count in retriever_miss_counter.most_common(5))),
        ("top_reranker_missed_enums", ", ".join(f"{enum_id} ({count})" for enum_id, count in reranker_miss_counter.most_common(5))),
    ],
    columns=["kpi", "value"],
)
display(executive_kpis)

## Root Cause And Team Analysis

Main prioritization views: judge root cause, issue categories, deterministic issues, combined affected teams, and cross-tabs.

In [0]:
def explode_counter(column: str) -> pd.DataFrame:
    counter = Counter(item for items in df[column] for item in items)
    return pd.DataFrame(counter.most_common(), columns=[column.strip("_"), "count"])


primary_root_cause_distribution = count_share(df["primary_root_cause"] if "primary_root_cause" in df.columns else pd.Series(["(missing)"] * len(df)), "rows")
issue_category_distribution = explode_counter("_issue_categories")
deterministic_issue_distribution = explode_counter("_deterministic_issues")
combined_team_distribution = explode_counter("_combined_affected_teams")

display(primary_root_cause_distribution)
display(issue_category_distribution)
display(deterministic_issue_distribution)
display(combined_team_distribution)

if {"primary_root_cause", "case_scope"}.issubset(df.columns):
    display(pd.crosstab(df["primary_root_cause"].fillna("(missing)").replace("", "(blank)"), df["case_scope"].fillna("(missing)").replace("", "(blank)")))

team_category_rows = []
for _, row in df.iterrows():
    for team in row["_combined_affected_teams"]:
        for category in row["_categories_list"]:
            team_category_rows.append({"team": team, "category": category})
affected_team_by_category = (
    pd.DataFrame(team_category_rows)
    .pipe(lambda x: pd.crosstab(x["team"], x["category"]) if len(x) else pd.DataFrame())
)
display(affected_team_by_category)

df["issue_combination"] = df.apply(
    lambda row: " + ".join(sorted(set(row["_issue_categories"] + row["_deterministic_issues"]))) or "no_issue",
    axis=1,
)
display(count_share(df["issue_combination"], "rows").head(25))

## Candidate-Pool Content And KB Editorial Analysis

Safe claim: the retrieved candidate pool did or did not contain adequate content under optimal selection. This is not a full-SKKB coverage audit.

In [0]:
kb_answerable = df["case_scope"] == "kb_answerable"
candidate_gap_rate = df.loc[kb_answerable, "candidate_pool_content_gap_identified_bool"].mean()
print(f"candidate-pool content gap rate among kb_answerable cases: {candidate_gap_rate:.1%}")

display(count_share(df["optimal_retrieved_context_adequacy_score"], "rows"))

if "kb_change_type" in df.columns:
    display(count_share(df["kb_change_type"], "rows"))

optimal_enum_counter = Counter(
    enum_id
    for enums in df["_optimal_enum_selection"]
    for enum_id in enums
)
top_optimal_enums = pd.DataFrame(optimal_enum_counter.most_common(25), columns=["enum_id", "count"])
display(top_optimal_enums)

gap_cols = [
    "test_case_id",
    "user_query",
    "case_scope",
    "candidate_pool_content_gap_description",
    "kb_change_type",
    "kb_improvement_suggestion",
    "operational_verdict",
]
gap_cols = [col for col in gap_cols if col in df.columns]
candidate_pool_gap_examples = (
    df[df["candidate_pool_content_gap_identified_bool"]]
    .sort_values(["quality_gate", "judge_weighted_avg"], ascending=[True, True])
    [gap_cols]
)
display(candidate_pool_gap_examples.head(25))

kb_suggestion_counts = count_share(
    df["kb_improvement_suggestion"].fillna("").replace("", "(blank)"),
    "rows",
)
display(kb_suggestion_counts.head(20))

## Retrieval And Reranker Analysis

Missed expected ENUMs are split by retriever misses and reranker misses.

In [0]:
top_retriever_missed = pd.DataFrame(retriever_miss_counter.most_common(50), columns=["enum_id", "retriever_miss_count"])
top_reranker_missed = pd.DataFrame(reranker_miss_counter.most_common(50), columns=["enum_id", "reranker_miss_count"])
missed_enum_comparison = (
    pd.merge(top_retriever_missed, top_reranker_missed, on="enum_id", how="outer")
    .fillna(0)
    .assign(total_miss_count=lambda x: x["retriever_miss_count"] + x["reranker_miss_count"])
    .sort_values("total_miss_count", ascending=False)
)

display(missed_enum_comparison.head(50))

retrieved_low_rank_rows = []
for _, row in df.iterrows():
    ranks = rank_map(row["_retrieved_enum_ids"])
    for enum_id in row["_expected_enums"]:
        rank = ranks.get(enum_id)
        if rank and rank > LOW_RANK_THRESHOLD:
            retrieved_low_rank_rows.append(
                {
                    "test_case_id": row["test_case_id"],
                    "enum_id": enum_id,
                    "rank": rank,
                    "user_query": row.get("user_query"),
                    "retrieval_recall": row["retrieval_recall"],
                    "answer_expected_alignment_score": row["answer_expected_alignment_score"],
                }
            )
retrieved_low_rank_expected_enums = pd.DataFrame(retrieved_low_rank_rows)
display(retrieved_low_rank_expected_enums.head(50))

example_cols = [
    "test_case_id",
    "user_query",
    "expected_enums",
    "retrieved_enum_ids",
    "reranked_enum_ids",
    "retrieval_recall",
    "reranker_conditional_recall",
    "selection_f1",
    "selection_semantic_relevance_score",
    "selected_context_sufficiency_score",
    "answer_expected_alignment_score",
    "operational_verdict",
    "overall_explanation",
]
example_cols = [col for col in example_cols if col in df.columns]

retriever_succeeded_reranker_failed = df[
    (df["retrieval_recall"] >= 0.999)
    & (df["reranker_conditional_recall"] < 0.999)
][example_cols]
retriever_failed_before_reranker = df[
    df["retriever_missed_expected_count"] > 0
][example_cols]
exact_set_good_semantic_low = df[
    (df["selection_f1"] >= 0.999)
    & (df["selection_semantic_relevance_score"] < 2)
][example_cols]

display(Markdown("### Retriever succeeded but reranker failed"))
display(retriever_succeeded_reranker_failed.head(20))
display(Markdown("### Retriever failed before reranker had a chance"))
display(retriever_failed_before_reranker.head(20))
display(Markdown("### Exact set metric good but semantic selection score low"))
display(exact_set_good_semantic_low.head(20))

## Answer Quality And Grounding Analysis

Separates answer-generation failures from upstream selected-context insufficiency.

In [0]:
display(count_share(df["answer_expected_alignment_score"], "rows"))
display(count_share(df["answer_groundedness_score"], "rows"))

missing_fact_counter = Counter(
    item
    for items in df["_missing_facts"]
    for item in items
)
hallucinated_claim_counter = Counter(
    item
    for items in df["_hallucinated_claims"]
    for item in items
)

display(pd.DataFrame(missing_fact_counter.most_common(50), columns=["missing_fact", "count"]))
display(pd.DataFrame(hallucinated_claim_counter.most_common(50), columns=["hallucinated_claim", "count"]))

df["answer_generation_issue_isolated"] = (
    (df["answer_expected_alignment_score"] <= 1)
    & (df["selected_context_sufficiency_score"] == 2)
)
df["answer_failure_with_insufficient_context"] = (
    (df["answer_expected_alignment_score"] <= 1)
    & (df["selected_context_sufficiency_score"] <= 1)
)

answer_split_summary = pd.DataFrame(
    {
        "split": [
            "answer_generation_issue_isolated",
            "answer_failure_with_insufficient_context",
            "grounding_failure",
        ],
        "rows": [
            df["answer_generation_issue_isolated"].sum(),
            df["answer_failure_with_insufficient_context"].sum(),
            (df["answer_groundedness_score"] <= 1).sum(),
        ],
    }
)
answer_split_summary["share"] = (answer_split_summary["rows"] / len(df)).round(4)
display(answer_split_summary)

answer_failure_cols = [
    "test_case_id",
    "user_query",
    "selected_context_sufficiency_score",
    "answer_expected_alignment_score",
    "answer_groundedness_score",
    "missing_facts",
    "hallucinated_claims",
    "bot_improvement_suggestion",
    "overall_explanation",
]
answer_failure_cols = [col for col in answer_failure_cols if col in df.columns]
display(df[df["answer_generation_issue_isolated"]][answer_failure_cols].head(25))

## Language And Agent Prompt Analysis

Language compliance is blocking for Slovak deployment quality.

In [0]:
display(count_share(df["language_compliance_score"], "rows"))

df["language_mismatch"] = df["language_compliance_score"] < 2
print(
    f"language mismatch rate: {df['language_mismatch'].mean():.1%} "
    f"({int(df['language_mismatch'].sum())}/{len(df)})"
)

language_evidence_counter = Counter(
    item
    for items in df["_bot_language_issue_evidence"]
    for item in items
)
display(pd.DataFrame(language_evidence_counter.most_common(50), columns=["language_issue_evidence", "count"]))

language_cols = [
    "test_case_id",
    "user_query",
    "bot_response",
    "language_compliance_score",
    "bot_language_issue_evidence",
    "bot_improvement_suggestion",
    "overall_explanation",
]
language_cols = [col for col in language_cols if col in df.columns]
display(df[df["language_mismatch"]][language_cols].head(25))

## Correlation And Calibration

Uses pandas/numpy only: Pearson on raw values and Spearman on ranks.

In [0]:
def correlation_row(x: pd.Series, y: pd.Series, label: str) -> dict:
    pair = pd.concat([x, y], axis=1).dropna()
    if len(pair) < 3:
        return {"pair": label, "n": len(pair), "pearson": np.nan, "spearman": np.nan}
    x_values = pair.iloc[:, 0]
    y_values = pair.iloc[:, 1]
    return {
        "pair": label,
        "n": len(pair),
        "pearson": x_values.corr(y_values, method="pearson"),
        "spearman": x_values.rank(method="average").corr(y_values.rank(method="average"), method="pearson"),
    }


correlation_rows = []
if "expert_score" in df.columns:
    correlation_rows.append(correlation_row(df["judge_weighted_avg"], df["expert_score"], "judge_weighted_avg vs expert_score"))
if "enum_relevance_score" in df.columns:
    correlation_rows.append(correlation_row(df["selection_f1"], df["enum_relevance_score"], "selection_f1 vs enum_relevance_score"))
    correlation_rows.append(correlation_row(df["retrieval_recall"], df["enum_relevance_score"], "retrieval_recall vs enum_relevance_score"))

correlation_table = pd.DataFrame(correlation_rows).round(4)
display(correlation_table)

df["judge_weighted_avg_bin"] = pd.cut(
    df["judge_weighted_avg"],
    bins=[-np.inf, 0.5, 1.0, PASS_THRESHOLD, np.inf],
    labels=["bad", "weak", "mid", "pass"],
    right=False,
)
if "expert_score" in df.columns:
    df["expert_score_bin"] = pd.cut(
        df["expert_score"],
        bins=[-np.inf, 3, 6, 8, np.inf],
        labels=["low", "mid", "good", "excellent"],
        right=True,
    )
    display(pd.crosstab(df["expert_score_bin"], df["judge_weighted_avg_bin"], margins=True))

df["selection_f1_bin"] = pd.cut(
    df["selection_f1"],
    bins=[-np.inf, 0, 0.5, 1.0, np.inf],
    labels=["none", "weak", "mid", "perfect"],
    right=False,
)
df.loc[df["selection_f1"].eq(1.0), "selection_f1_bin"] = "perfect"
if "enum_relevance_score" in df.columns:
    df["enum_relevance_bin"] = pd.cut(
        df["enum_relevance_score"],
        bins=[-np.inf, 0.33, 0.66, 0.9, np.inf],
        labels=["low", "mid", "good", "excellent"],
        right=True,
    )
    display(pd.crosstab(df["enum_relevance_bin"], df["selection_f1_bin"], margins=True))

if "expert_score" in df.columns:
    df["_judge_expert_gap"] = ((df["judge_weighted_avg"] / 2.0) - ((df["expert_score"] - 1) / 9.0)).abs()
    disagreement_cols = [
        "test_case_id",
        "user_query",
        "judge_weighted_avg",
        "expert_score",
        "_judge_expert_gap",
        "operational_verdict",
        "primary_root_cause",
        "overall_explanation",
    ]
    disagreement_cols = [col for col in disagreement_cols if col in df.columns]
    display(df.sort_values("_judge_expert_gap", ascending=False)[disagreement_cols].head(25))

## Optional Charts

The computations above do not require Plotly. If Plotly is installed in your notebook environment, this cell renders the recommended quick charts.

In [0]:
try:
    import plotly.express as px
except ImportError:
    px = None

if px is None:
    print("Plotly is not installed in this environment. Use the tables above, or install plotly to render charts.")
else:
    fig = px.histogram(
        df,
        x="judge_weighted_avg",
        nbins=24,
        title=f"Judge weighted average (pass threshold = {PASS_THRESHOLD})",
    )
    fig.add_vline(x=PASS_THRESHOLD, line_dash="dash", line_color="red")
    fig.show()

    px.histogram(df, x="retrieval_recall", nbins=20, title="Retrieval recall").show()
    px.histogram(df, x="weighted_retrieval_recall", nbins=20, title="Weighted retrieval recall").show()
    px.histogram(df, x="selection_f1", nbins=20, title="Selection F1").show()
    px.scatter(
        df,
        x="retrieval_recall",
        y="answer_expected_alignment_score",
        color="operational_verdict",
        hover_data=["test_case_id", "user_query"],
        title="Retrieval recall vs answer expected alignment",
    ).show()
    px.scatter(
        df,
        x="selection_f1",
        y="answer_expected_alignment_score",
        color="operational_verdict",
        hover_data=["test_case_id", "user_query"],
        title="Selection F1 vs answer expected alignment",
    ).show()

## Final Stakeholder Tables

These are the concrete tables called out in the analysis plan.

In [0]:
stakeholder_tables = {
    "executive_kpis": executive_kpis,
    "dimension_score_distribution": dimension_distribution,
    "deterministic_metric_summary": metric_summary,
    "operational_verdict_distribution": count_share(df["operational_verdict"], "rows"),
    "primary_root_cause_distribution": primary_root_cause_distribution,
    "combined_affected_team_distribution": combined_team_distribution,
    "top_retriever_missed_enums": top_retriever_missed,
    "top_reranker_missed_enums": top_reranker_missed,
    "top_candidate_pool_content_gaps": candidate_pool_gap_examples,
    "top_bot_agent_issues": df[
        df["_combined_affected_teams"].apply(lambda teams: "bot_agent" in teams)
    ][
        [
            col
            for col in [
                "test_case_id",
                "user_query",
                "operational_verdict",
                "primary_root_cause",
                "answer_expected_alignment_score",
                "answer_groundedness_score",
                "language_compliance_score",
                "bot_improvement_suggestion",
                "overall_explanation",
            ]
            if col in df.columns
        ]
    ],
    "judge_consistency_errors": df[df["validation_error_count"] > 0][
        [
            col
            for col in [
                "test_case_id",
                "user_query",
                "case_scope",
                "operational_verdict",
                "validation_errors",
                "validation_warnings",
                "overall_explanation",
            ]
            if col in df.columns
        ]
    ],
}

for name, table in stakeholder_tables.items():
    display(Markdown(f"### {name}"))
    display(table.head(25) if isinstance(table, pd.DataFrame) else table)

## Manual Review Queue

Sorted by validation errors, multi-team blockers, answer quality, language mismatch, high-weight expected ENUM misses, and candidate-pool gaps.

In [0]:
def missed_expected_weight(row) -> float:
    missing = set(row["_expected_enums_missed_by_retriever"]) | set(row["_expected_enums_missed_by_reranker"])
    weights = row["_expected_enum_weights"]
    return sum(weights.get(enum_id, 1.0) for enum_id in missing)


df["missed_expected_weight"] = df.apply(missed_expected_weight, axis=1)
df["review_priority_score"] = (
    df["validation_error_count"] * 1000
    + df["operational_verdict"].eq("multi_team_blocker").astype(int) * 500
    + (df["answer_expected_alignment_score"] <= 1).astype(int) * 200
    + (df["answer_groundedness_score"] <= 1).astype(int) * 150
    + df["language_mismatch"].astype(int) * 125
    + df["missed_expected_weight"].fillna(0) * 20
    + df["candidate_pool_content_gap_identified_bool"].astype(int) * 50
)

manual_review_cols = [
    "test_case_id",
    "user_query",
    "case_scope",
    "operational_verdict",
    "primary_root_cause",
    "issue_categories",
    "combined_affected_teams",
    "judge_weighted_avg",
    "retrieval_recall",
    "selection_f1",
    "answer_expected_alignment_score",
    "answer_groundedness_score",
    "language_compliance_score",
    "validation_errors",
    "deterministic_issues",
    "missed_expected_weight",
    "retrieval_improvement_suggestion",
    "reranker_improvement_suggestion",
    "kb_improvement_suggestion",
    "bot_improvement_suggestion",
    "test_case_improvement_suggestion",
]
manual_review_cols = [col for col in manual_review_cols if col in df.columns]
manual_review_queue = (
    df.sort_values(
        [
            "review_priority_score",
            "validation_error_count",
            "judge_weighted_avg",
        ],
        ascending=[False, False, True],
    )[manual_review_cols]
)
display(manual_review_queue.head(100))

## Recommendation Inputs

This cell turns the highest-volume findings into action buckets. Treat it as evidence for the final written report, not as an automatic final recommendation.

In [0]:
recommendation_inputs = {
    "retrieval": {
        "top_missed_enums": top_retriever_missed.head(10),
        "low_rank_expected_enums": retrieved_low_rank_expected_enums.head(10),
        "suggestions": count_share(df["retrieval_improvement_suggestion"].fillna("").replace("", "(blank)"), "rows").head(10),
    },
    "reranker": {
        "top_missed_enums": top_reranker_missed.head(10),
        "invalid_output_rows": df[df["invalid_selected_enum_count"] > 0][["test_case_id", "reranker_invalid_selected_ids", "reranker_selection_status"]],
        "suggestions": count_share(df["reranker_improvement_suggestion"].fillna("").replace("", "(blank)"), "rows").head(10),
    },
    "kb_editors": {
        "candidate_pool_gaps": candidate_pool_gap_examples.head(10),
        "kb_change_types": count_share(df["kb_change_type"] if "kb_change_type" in df.columns else pd.Series(["(missing)"] * len(df)), "rows"),
        "suggestions": count_share(df["kb_improvement_suggestion"].fillna("").replace("", "(blank)"), "rows").head(10),
    },
    "bot_agent": {
        "answer_generation_issues": df[df["answer_generation_issue_isolated"]][answer_failure_cols].head(10),
        "language_issues": df[df["language_mismatch"]][language_cols].head(10),
        "suggestions": count_share(df["bot_improvement_suggestion"].fillna("").replace("", "(blank)"), "rows").head(10),
    },
    "routing_tooling": {
        "kb_without_search": df[df["kb_case_without_knowledge_search"]][["test_case_id", "user_query", "tools_called"]].head(10),
        "empty_kb_context": df[df["empty_kb_context_for_kb_case"]][["test_case_id", "user_query", "tools_called"]].head(10),
    },
    "test_set": {
        "non_kb_or_unclear_scope": df[df["exclude_from_kb_content_conclusions"]][["test_case_id", "user_query", "case_scope", "test_case_improvement_suggestion"]].head(20),
        "suggestions": count_share(df["test_case_improvement_suggestion"].fillna("").replace("", "(blank)"), "rows").head(10),
    },
}

for bucket, tables in recommendation_inputs.items():
    display(Markdown(f"### {bucket}"))
    for name, table in tables.items():
        display(Markdown(f"**{name}**"))
        display(table)

## Optional Export

Set `EXPORT_ENRICHED = True` in the parameters cell to write an enriched CSV next to the input checkpoint.

In [0]:
if EXPORT_ENRICHED:
    export_path = checkpoint_path.with_name(f"{checkpoint_path.stem}_final_analysis_enriched.csv")
    export_df = df.copy()
    for col in list(export_df.columns):
        if col.startswith("_"):
            export_df = export_df.drop(columns=[col])
    export_df.to_csv(export_path, index=False)
    print(f"wrote enriched analysis CSV: {export_path}")
else:
    print("EXPORT_ENRICHED is False; no file written.")